# GUI Build
This is a test build of the GUI before we successfully integrated the spectrometer. It uses data in the same form as the spectrometer generates for testing purposes, but does not link to the spectrometer yet.

### Some notes:
- once the spectrometer is linked, the thread running _live_loop may need to be "inverted" (since the delay will happen after requesting spectrometer data)
- the file saving system still needs to be worked out
- eventually, the GUI class should be linked to a Spectrometer from seabreeze

In [1]:
import time
import threading 
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

%matplotlib widget

rng = np.random.default_rng()

def noisyGaussian(x):
    '''A quick function that simulates what a live might look like in the PLQY.'''
    output = np.zeros_like(x)
    center = np.mean(x)
    width = np.std(x)/2
    output += np.exp( -((x-center)/width)**2  )
    output += rng.random(len(x))*0.1
    return output 

xx = np.linspace(250, 800, 1024) #test data

class PL_GUI: #The class that contains the GUI. Currently pairs to X data; will eventually pair to a spectrometer
    #(In the future, this means we can have 2 GUIs easily open for absorbance studies!)
    def __init__(self, xx):
        self.xx = xx
        
        # initalize variables
        self.live = False 
        self.loop_thread = None  

        # create widgets for inputs, etc.
        self.exposure = widgets.FloatText(description='Exposure (ms)', value=200)
        self.noOfAv_input = widgets.IntText(description='Averages', value=1)
        self.modeSelector = widgets.Button(description='Start live')
        self.saveFig = widgets.Button(description='Save current figure')
        
        #initalize a text output window for a console
        self.text_output = widgets.Textarea(
            value='GUI Initialized.\n',
            disabled=True,
            layout=widgets.Layout(height='120px', width='100%')
        )

        # link widgets
        self.modeSelector.on_click(self.onModeSwap)
        self.exposure.observe(self.onExposureUpdate, names='value')
        self.noOfAv_input.observe(self.onAvUpdate, names='value')
        self.saveFig.on_click(self.onFigSaveRequested)

        self.exposureTime = self.exposure.value
        self.noOfAv = self.noOfAv_input.value
        
        #create the plot
        plt.ioff() #disable standard behavior; prevents two plots from appearing
        plt.close('all') 
        self.fig, self.ax = plt.subplots(figsize=(6, 4)) #create the plot
        self.ax.set_xlabel('Wavelength (nm)')
        self.ax.set_ylabel('Intensity (a.u.)')
        self.ax.set_ylim(-0.1, 1.2)
        self.data, = self.ax.plot(self.xx, noisyGaussian(self.xx), label='Live') #populate inital data
        self.ax.legend()
        plt.ion() # Restore standard behavior

        #create the GUI layout
        self.layout = widgets.HBox([ 
            self.fig.canvas, 
            widgets.VBox([
                self.exposure,
                self.noOfAv_input,
                self.modeSelector,
                self.saveFig,
                self.text_output
            ])
        ])

    def show(self):
        '''Wrapper to display the GUI.'''
        display(self.layout)

    def log(self, message, clear=False):
        '''Helper function to push text to console.'''
        if clear:
            self.text_output.value = ""
        self.text_output.value += f"{message}\n"

    def onModeSwap(self, button):
        '''When the mode swap button is pressed:'''
        if self.live:
            #self.log('Live feed terminating...') #debugging
            self.stopLiveFeed()
            button.description = 'Start live'
        else:
            #self.log('Starting feed...', clear=True) #debugging
            self.startLiveFeed()
            button.description = 'Stop live'

    def startLiveFeed(self):
        '''Function to start the live feed in the background via a Thread.'''
        self.live = True
        self.loop_thread = threading.Thread(target=self._live_loop, daemon=True) #the thread must be a daemon to stop it from running in the background
        #self.log("A thread has been initialized.") #debugging
        self.loop_thread.start()

    def stopLiveFeed(self):
        '''For parity with starting, a wrapper function that disables the live feed.'''
        self.live = False #we let the thread die by itself, to avoid issues with join()

    def _live_loop(self):
        '''The drawing loop that runs in the background. Eventally, will also request data from the spectrometer and draw it.'''
        while self.live:
            newY = np.zeros_like(self.xx) #since spectrometer not yet connected, we simulate data
            for _ in range(self.noOfAv):
                newY += noisyGaussian(self.xx)
            newY /= self.noOfAv #simulate averaging
            self.data.set_ydata(newY) #set the new ydata
            
            self.fig.canvas.draw_idle() #draw the data to the canvas
            
            time.sleep((self.exposureTime/1000)*self.noOfAv) #simulate the time required for the spectrometer to collect the dataset

    def onExposureUpdate(self, change):
        ''' Function to update the exposure time (.value may cause issues with threading)'''
        self.exposureTime = change['new']
    
    def onAvUpdate(self, change):
        ''' Function to update the number of spectrums averaged (.value may cause issues with threading)'''
        self.noOfAv = change['new']

    def onFigSaveRequested(self, button_object):
        '''Will eventually save a spectrum to a csv and pin the spectrum to the GUI for future reference'''
        self.log('Still a WIP!', clear=True)

In [2]:
gui = PL_GUI(xx)
gui.show()